In [1]:
from pathlib import Path
import os
import sys
import time

import torch
import hydra
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader

PROJECT_ROOT = Path(
    "/homes/cmininno/tesi/surgical_scene_segmentation"
)

CONFIG_DIR = PROJECT_ROOT / "configs"
CONFIG_NAME = "config"

# Questa lista corrisponde agli override della riga Slurm/CLI.
#
# Esempio:
# python start_experiment.py \
#   dataset=baseline/cholec_seg8k \
#   training=baseline/cholec_seg8k \
#   model=baseline/mask2former \
#   train.optimizer.lr=1e-3

OVERRIDES = [
    "dataset=baseline/sarrarp50",
    "training=baseline/sarrarp50",
    "model=baseline/unet",
    "train.optimizer.lr=1e-3",

    # Debug locale: evita di aspettare 50 epoche per scoprire un bug.
    "train.max_epochs=1",
    "logging.wandb.enabled=false",
]

# True = esegue solo la costruzione + sanity check della pipeline.
# False = esegue anche Trainer.fit().
SANITY_CHECK_ONLY = False

# Se vuoi eseguire davvero il training dal notebook:
# SANITY_CHECK_ONLY = False

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project: {PROJECT_ROOT}")
print(f"Config : {CONFIG_DIR}")
print("Overrides:")
for override in OVERRIDES:
    print(f"  - {override}")


Project: /homes/cmininno/tesi/surgical_scene_segmentation
Config : /homes/cmininno/tesi/surgical_scene_segmentation/configs
Overrides:
  - dataset=baseline/sarrarp50
  - training=baseline/sarrarp50
  - model=baseline/unet
  - train.optimizer.lr=1e-3
  - train.max_epochs=1
  - logging.wandb.enabled=false


In [2]:
# ============================================================
# 1. Hydra: stessa composizione usata dal job
# ============================================================

GlobalHydra.instance().clear()

with hydra.initialize_config_dir(
    version_base=None,
    config_dir=str(CONFIG_DIR),
):
    cfg = compose(
        config_name=CONFIG_NAME,
        overrides=OVERRIDES,
    )

print(OmegaConf.to_yaml(cfg, resolve=False))


name: sarrarp50
data:
  name: sarrarp50
  dataset:
    _target_: src.datasets.sarrarp50_dataset.SARRARP50Dataset
    root: ${paths.data}/SARRARP50
  num_classes: 10
  split:
    strategy: official
    stratified: false
    group_by: video
preprocessing:
  image:
    size:
    - 480
    - 640
    channel_first: true
    dtype: float32
    normalize: true
  mask:
    channel_first: true
    dtype: long
augmentation:
  train:
    enabled: true
    scaling:
      enabled: true
      range:
      - -0.1
      - 0.1
      probability: 0.3
    rotation:
      enabled: true
      range:
      - -10
      - 10
      probability: 0.3
    translation:
      enabled: true
      range:
      - -0.05
      - 0.05
      probability: 0.3
    horizontal_flip:
      enabled: true
      probability: 0.5
    vertical_flip:
      enabled: false
    color_jitter:
      enabled: true
      probability: 0.3
      brightness: 0.2
      contrast: 0.2
      saturation: 0.2
      hue: 0.05
  validation:
    enabl

In [3]:
# ============================================================
# 2. Setup
# ============================================================

from src.utils.repro import set_seed, seed_worker

experiment_start = time.time()

set_seed(cfg.experiment.seed)

generator = torch.Generator()
generator.manual_seed(cfg.experiment.seed)

device = torch.device(
    cfg.device if torch.cuda.is_available() else "cpu"
)

print(f"Seed   : {cfg.experiment.seed}")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


Seed   : 42
Device : cuda
GPU    : Quadro RTX 5000


In [4]:
# ============================================================
# 3. Dataset
# ============================================================

from src.datasets.transformation.factory import TransformFactory
from src.datasets.transformation.transform_dataset import TransformDataset

print(f"Dataset: {cfg.data.name}")

dataset = instantiate(cfg.data.dataset)

print(f"Dataset class : {dataset.__class__.__name__}")
print(f"Dataset size  : {len(dataset)}")
print(f"Num classes   : {dataset.num_classes}")
print(f"Class names   : {dataset.CLASS_NAMES}")


/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: sarrarp50
Dataset class : SARRARP50Dataset
Dataset size  : 16295
Num classes   : 10
Class names   : ['Background', 'Tool clasper', 'Tool wrist', 'Tool shaft', 'Suturing needle', 'Thread', 'Suction tool', 'Needle Holder', 'Clamps', 'Catheter']


In [5]:
# ============================================================
# 4. Dataset split
# ============================================================

split_cfg = cfg.data.split

split_kwargs = {
    "strategy": split_cfg.strategy,
    "group_by": split_cfg.group_by,
}

if split_cfg.strategy == "percentage":
    split_kwargs.update(
        {
            "train_ratio": split_cfg.train_ratio,
            "val_ratio": split_cfg.val_ratio,
            "test_ratio": split_cfg.test_ratio,
            "seed": split_cfg.get(
                "seed",
                cfg.experiment.seed,
            ),
            "stratified": split_cfg.get(
                "stratified",
                False,
            ),
        }
    )

print("Split configuration:")
print(OmegaConf.to_yaml(split_cfg))

train_dataset, val_dataset, test_dataset = dataset.create_split(
    **split_kwargs
)

print()
print(f"Train samples: {len(train_dataset) if train_dataset else 0}")
print(f"Val samples  : {len(val_dataset) if val_dataset else 0}")
print(f"Test samples : {len(test_dataset) if test_dataset else 0}")


Split configuration:
strategy: official
stratified: false
group_by: video


DATASET SPLIT

Train:
  Groups : ['video_01', 'video_02', 'video_03', 'video_04', 'video_05', 'video_06', 'video_07', 'video_08', 'video_09', 'video_10', 'video_11_1', 'video_11_2', 'video_12', 'video_13', 'video_14', 'video_15_1', 'video_15_2', 'video_16', 'video_17_1', 'video_17_2', 'video_18', 'video_19', 'video_20', 'video_21', 'video_22', 'video_23', 'video_24', 'video_25', 'video_26', 'video_27', 'video_28', 'video_29_1', 'video_29_2', 'video_30', 'video_31', 'video_32', 'video_33', 'video_34', 'video_35', 'video_36', 'video_37', 'video_38', 'video_39', 'video_40']
  Frames : 13043

Validation:
  Groups : ['video_41', 'video_42', 'video_43', 'video_44', 'video_45', 'video_46', 'video_47', 'video_48', 'video_49', 'video_50']
  Frames : 3252

Total:
  Groups : 54
  Frames : 16295

Train samples: 13043
Val samples  : 3252
Test samples : 0


In [6]:
# ============================================================
# 5. Transforms
# ============================================================

transform_factory = TransformFactory(cfg)

train_transform = transform_factory.build_train_transform()
val_transform = transform_factory.build_val_transform()

test_transform = (
    transform_factory.build_test_transform()
    if test_dataset is not None
    else None
)

if train_dataset is not None:
    train_dataset = TransformDataset(
        train_dataset,
        transform=train_transform,
    )

if val_dataset is not None:
    val_dataset = TransformDataset(
        val_dataset,
        transform=val_transform,
    )

if test_dataset is not None:
    test_dataset = TransformDataset(
        test_dataset,
        transform=test_transform,
    )

print("Transforms and TransformDataset wrappers created.")


Transforms and TransformDataset wrappers created.


In [7]:
# ============================================================
# 6. DataLoaders
# ============================================================

train_loader = None
val_loader = None
test_loader = None

if train_dataset is not None:
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=True,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

if val_dataset is not None:
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=False,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

if test_dataset is not None:
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.train.batch_size,
        shuffle=False,
        num_workers=cfg.train.num_workers,
        pin_memory=cfg.train.pin_memory,
        worker_init_fn=seed_worker,
        generator=generator,
    )

print(f"Train batches: {len(train_loader) if train_loader else 0}")
print(f"Val batches  : {len(val_loader) if val_loader else 0}")
print(f"Test batches : {len(test_loader) if test_loader else 0}")


Train batches: 3261
Val batches  : 813
Test batches : 0


In [8]:
# ============================================================
# 7. Inspect one transformed batch
# ============================================================

batch = next(iter(val_loader))

print("Batch keys:")
print(list(batch.keys()))

for key, value in batch.items():
    if torch.is_tensor(value):
        print(
            f"{key:>8}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}"
        )
    else:
        print(f"{key:>8}: type={type(value).__name__}")


Batch keys:
['image', 'mask', 'video', 'frame']
   image: shape=(4, 3, 480, 640), dtype=torch.float32
    mask: shape=(4, 1, 480, 640), dtype=torch.int64
   video: type=list
   frame: shape=(4,), dtype=torch.int64


In [9]:
from __future__ import annotations

import torch
import torch.nn.functional as F

from transformers import (
    AutoImageProcessor,
    Mask2FormerConfig,
    Mask2FormerForUniversalSegmentation,
)


class Mask2FormerWrapper(torch.nn.Module):
    """Wrapper for Mask2Former semantic segmentation."""

    def __init__(
        self,
        checkpoint: str | None,
        num_labels: int,
    ) -> None:
        super().__init__()

        self.is_mask2former = True
        self.num_labels = num_labels

        # =========================================================
        # PROCESSOR
        # =========================================================

        if checkpoint is not None:
            self.processor = AutoImageProcessor.from_pretrained(
                checkpoint,
            )
        else:
            # No checkpoint: create a default processor from config.
            self.processor = AutoImageProcessor.from_pretrained(
                "facebook/mask2former-swin-base-ade-semantic",
            )

        # =========================================================
        # MODEL
        # =========================================================

        if checkpoint is not None:
            self.model = (
                Mask2FormerForUniversalSegmentation.from_pretrained(
                    checkpoint,
                    num_labels=num_labels,
                    ignore_mismatched_sizes=True,
                )
            )

        else:
            config = Mask2FormerConfig(
                num_labels=num_labels,
            )

            self.model = Mask2FormerForUniversalSegmentation(
                config,
            )

    def _build_targets(
        self,
        target: torch.Tensor,
    ) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        """Convert semantic masks into Mask2Former targets."""

        mask_labels = []
        class_labels = []

        for mask in target:

            # [1, H, W] -> [H, W]
            mask = mask.squeeze(0)

            masks = []
            classes = []

            unique_classes = torch.unique(mask)

            for class_id in unique_classes:

                class_id = int(class_id)

                # Background is not represented as an object mask.
                if class_id == 0:
                    continue

                binary_mask = mask == class_id

                if binary_mask.any():
                    masks.append(binary_mask)
                    classes.append(class_id)

            if masks:
                masks = torch.stack(masks).float()

                classes = torch.tensor(
                    classes,
                    dtype=torch.long,
                    device=mask.device,
                )

            else:
                masks = torch.empty(
                    (
                        0,
                        mask.shape[-2],
                        mask.shape[-1],
                    ),
                    dtype=torch.float32,
                    device=mask.device,
                )

                classes = torch.empty(
                    (0,),
                    dtype=torch.long,
                    device=mask.device,
                )

            mask_labels.append(masks)
            class_labels.append(classes)

        return mask_labels, class_labels

    def _post_process(
        self,
        outputs,
        target_sizes: list[tuple[int, int]],
    ) -> torch.Tensor:
        """Convert Mask2Former outputs into [B, C, H, W] predictions."""

        predictions = self.processor.post_process_semantic_segmentation(
            outputs,
            target_sizes=target_sizes,
        )

        predictions = torch.stack(predictions)

        predictions = F.one_hot(
            predictions.long(),
            num_classes=self.num_labels,
        )

        predictions = predictions.permute(
            0,
            3,
            1,
            2,
        ).float()

        return predictions

    def forward(
        self,
        images: torch.Tensor,
        masks: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Forward pass.

        Returns:
            predictions:
                Tensor [B, C, H, W].

            loss:
                Mask2Former loss during training, otherwise None.
        """

        target_sizes = [
            (images.shape[-2], images.shape[-1])
        ] * images.shape[0]

        # =========================================================
        # TRAINING
        # =========================================================

        if self.training:

            if masks is None:
                raise ValueError(
                    "Target masks are required during training."
                )

            mask_labels, class_labels = self._build_targets(
                masks,
            )

            outputs = self.model(
                pixel_values=images,
                mask_labels=mask_labels,
                class_labels=class_labels,
            )

            loss = outputs.loss

        # =========================================================
        # VALIDATION / TEST
        # =========================================================

        else:

            outputs = self.model(
                pixel_values=images,
            )

            loss = None

        # =========================================================
        # SEMANTIC SEGMENTATION OUTPUT
        # =========================================================

        predictions = self._post_process(
            outputs,
            target_sizes,
        )

        return predictions, loss

In [10]:
from pathlib import Path
import sys

print("Python:", sys.executable)
print("Working directory:", Path.cwd())
print("src exists:", (Path.cwd() / "src").exists())

Python: /homes/cmininno/tesi/tesi_venv/bin/python
Working directory: /homes/cmininno/tesi/surgical_scene_segmentation
src exists: True


In [11]:
import sys
import importlib.util

print("Python:", sys.executable)
print("nnunetv2:", importlib.util.find_spec("nnunetv2"))
print("umamba_adapter:", importlib.util.find_spec("src.models.umamba_adapter"))

Python: /homes/cmininno/tesi/tesi_venv/bin/python
nnunetv2: None
umamba_adapter: ModuleSpec(name='src.models.umamba_adapter', loader=<_frozen_importlib_external.SourceFileLoader object at 0x146b7c845410>, origin='/homes/cmininno/tesi/surgical_scene_segmentation/src/models/umamba_adapter.py')


In [12]:
# ============================================================
# 8. Model
# ============================================================

model = instantiate(cfg.model.architecture).to(device)


#num_classes = cfg.data.num_classes
#
#model = Mask2FormerWrapper(
#    checkpoint=cfg.model.architecture.checkpoint,
#    num_labels=num_classes,
#)

print(f"Model class: {model.__class__.__name__}")
print(f"Mask2Former: {getattr(model, 'is_mask2former', False)}")

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Parameters         : {num_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")


Model class: UNet
Mask2Former: False
Parameters         : 6,501,961
Trainable parameters: 6,501,961


In [13]:
# ============================================================
# 9. Optimizer, scheduler, loss, scaler
# ============================================================

optimizer = instantiate(
    cfg.train.optimizer,
    params=model.parameters(),
)

scheduler = instantiate(
    cfg.train.scheduler,
    optimizer=optimizer,
)

loss_fn = instantiate(cfg.train.loss)

scaler = (
    instantiate(cfg.train.scaler)
    if "scaler" in cfg.train
    else None
)

print("Optimizer :", optimizer.__class__.__name__)
print("Scheduler :", scheduler.__class__.__name__)
print("Loss      :", loss_fn.__class__.__name__)
print("Scaler    :", scaler.__class__.__name__ if scaler else None)
print("LR        :", optimizer.param_groups[0]["lr"])


Optimizer : AdamW
Scheduler : CosineAnnealingLR
Loss      : DiceCELoss
Scaler    : GradScaler
LR        : 0.001


In [14]:
# ============================================================
# 10. Metric manager
# ============================================================

from src.metrics.metric_manager import MetricManager

metrics = {
    name: instantiate(metric_cfg)
    for name, metric_cfg in cfg.evaluation.metrics.items()
}

class_names = {
    index: name
    for index, name in enumerate(dataset.CLASS_NAMES)
}

metric_manager = MetricManager(
    metrics=metrics,
    num_classes=dataset.num_classes,
    dataset_name=cfg.data.name,
    class_names=class_names,
)

print("Metrics:", metric_manager.names)


Metrics: ['dice', 'iou', 'hd95']


In [15]:
# ============================================================
# 11. Sanity check: forward + shape contract
# ============================================================

def describe_tensor(name, tensor):
    if tensor is None:
        print(f"{name}: None")
        return

    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}, "
        f"device={tensor.device}"
    )


def check_segmentation_shape(
    predictions,
    masks,
    num_classes,
):
    if predictions is None:
        raise RuntimeError(
            "Model returned None instead of segmentation predictions."
        )

    if predictions.ndim != 4:
        raise ValueError(
            "Expected predictions with shape [B, C, H, W], "
            f"got {tuple(predictions.shape)}"
        )

    if masks.ndim not in (3, 4):
        raise ValueError(
            f"Unexpected target shape: {tuple(masks.shape)}"
        )

    # --------------------------------------------------------
    # Batch size
    # --------------------------------------------------------

    if predictions.shape[0] != masks.shape[0]:
        raise ValueError(
            "Batch size mismatch:\n"
            f"  predictions: {tuple(predictions.shape)}\n"
            f"  masks      : {tuple(masks.shape)}"
        )

    # --------------------------------------------------------
    # Number of classes
    # --------------------------------------------------------

    predicted_classes = predictions.shape[1]

    if predicted_classes != num_classes:
        raise ValueError(
            "CLASS COUNT MISMATCH:\n"
            f"  predictions: {predicted_classes}\n"
            f"  config     : {num_classes}"
        )

    # --------------------------------------------------------
    # Spatial dimensions
    # --------------------------------------------------------

    target_hw = tuple(masks.shape[-2:])
    prediction_hw = tuple(predictions.shape[-2:])

    if prediction_hw != target_hw:
        raise ValueError(
            "SEGMENTATION SHAPE MISMATCH:\n"
            f"  predictions : {tuple(predictions.shape)}\n"
            f"  target      : {tuple(masks.shape)}\n"
            f"  prediction spatial size = {prediction_hw}\n"
            f"  target spatial size     = {target_hw}"
        )


# ============================================================
# Batch
# ============================================================

images = batch["image"].to(device)
masks = batch["mask"].to(device)

num_classes = int(cfg.data.num_classes)

describe_tensor("images", images)
describe_tensor("masks", masks)


# ============================================================
# Inference forward
# ============================================================

model.eval()

with torch.no_grad():
    predictions, loss = model(images)

describe_tensor("predictions", predictions)
describe_tensor("loss", loss)


# ============================================================
# Shape validation
# ============================================================

check_segmentation_shape(
    predictions=predictions,
    masks=masks,
    num_classes=num_classes,
)

print("Inference shape sanity check: PASSED")

images: shape=(4, 3, 480, 640), dtype=torch.float32, device=cuda:0
masks: shape=(4, 1, 480, 640), dtype=torch.int64, device=cuda:0


/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/torch/_tensor.py:1386: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  ret = func(*args, **kwargs)


ValueError: too many values to unpack (expected 2)

In [ ]:
# ============================================================
# 12. Optional direct metric sanity check
# ============================================================

# Questo passaggio è volutamente separato dal training.
# Se fallisce, non ha senso lanciare 50 epoche su Slurm.

metric_manager.reset()

metric_manager.update(
    predictions.detach(),
    masks.detach(),
)

metric_preview = metric_manager.compute()

print("Metric sanity check: PASSED")

for name, value in metric_preview.items():
    print(f"{name:>10}: {value:.6f}")

In [16]:
# ============================================================
# 13. Trainer
# ============================================================

from src.engine.trainer import Trainer


In [19]:
import importlib

import src.engine.trainer as trainer

importlib.reload(trainer)

Trainer = trainer.Trainer

In [20]:

# Per replicare completamente start_experiment.py, abilita WandB
# e inizializza il logger nella cella successiva.
#
# Per il debug locale è più semplice usare logger=None.

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    scaler=scaler,
    metric_manager=metric_manager,
    cfg=cfg,
    device=device,
    logger=None,
)

print("Trainer initialized.")


Trainer initialized.


In [ ]:
print(torch.cuda.memory_summary(device=device, abbreviated=True))

In [21]:
# ============================================================
# 14. Training
# ============================================================

if SANITY_CHECK_ONLY:
    print(
        "SANITY_CHECK_ONLY=True: training skipped. "
        "Set it to False to execute Trainer.fit()."
    )
    train_results = None
else:
    train_results = trainer.fit()
    print("\nTraining completed.")
    print("Best epoch :", train_results["best_epoch"])
    print("Best score :", train_results["best_score"])
    print(
        "Best checkpoint:",
        train_results["best_checkpoint"],
    )


Epoch 1 [Train]:   0%|          | 0/3261 [00:00<?, ?it/s]/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/torch/nn/functional.py:3053: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:71.)
  return torch._C._nn.cross_entropy_loss(input, target, weight, _Reduction.get_enum(reduction), ignore_index, label_smoothing)
/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/monai/metrics/utils.py:333: UserWarning: the ground truth of class 6 is all 0, this may result in nan/inf distance.
  warnings.warn(
/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/monai/metrics/utils.py:333: UserWarning: the ground truth of class 7 is all 0, this may result in nan/inf dis

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 15. Load best checkpoint
# ============================================================

if train_results is not None:
    checkpoint_path = train_results["best_checkpoint"]

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(f"Loaded best checkpoint: {checkpoint_path}")


In [ ]:
# ============================================================
# 15. Load best checkpoint
# ============================================================

if train_results is not None:
    checkpoint_path = train_results["best_checkpoint"]

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print(f"Loaded best checkpoint: {checkpoint_path}")


In [ ]:
# ============================================================
# 16. Optional final test
# ============================================================

if train_results is not None and test_loader is not None:
    from src.engine.tester import Tester

    test_metrics = {
        name: instantiate(metric_cfg)
        for name, metric_cfg in cfg.evaluation.metrics.items()
    }

    test_metric_manager = MetricManager(
        metrics=test_metrics,
        num_classes=dataset.num_classes,
        dataset_name=cfg.data.name,
        class_names=class_names,
    )

    tester = Tester(
        model=model,
        test_loader=test_loader,
        metric_manager=test_metric_manager,
        loss_fn=loss_fn,
        cfg=cfg,
        device=device,
        logger=None,
    )

    test_results = tester.test()

    print("Final test results:")
    for name, value in test_results.get("metrics", {}).items():
        print(f"{name:>10}: {value:.6f}")
elif train_results is not None:
    print("No test split available; final test skipped.")


In [ ]:
# ============================================================
# 17. Final summary
# ============================================================

elapsed = time.time() - experiment_start

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Dataset       : {cfg.data.name}")
print(f"Model         : {cfg.model.name}")
print(f"Device        : {device}")
print(f"Elapsed       : {elapsed:.2f}s")

if train_results is not None:
    print(f"Best epoch    : {train_results['best_epoch']}")
    print(f"Best score    : {train_results['best_score']}")
    print(
        f"Checkpoint    : {train_results['best_checkpoint']}"
    )
else:
    print("Training       : skipped")
